# 02 预处理阶段（Phase 2）

这个 notebook 对应 `Agent.md` 里的 **Phase 2: Preprocessing**。当前版本不再只生成一套 prepared feature matrix，而是围绕两套固定数据口径同时产出 Phase 3 / Phase 4 可复用 artifact：

- `traditional_core`
- `traditional_plus_proxy`

这里的 `proxy` 仅指 **Home Credit `application_train` 主表内部已有的 proxy 变量**，不代表外部另类数据。


## 1. Phase gate、运行参数与辅助函数

先确认原始数据路径和 `application_train.csv` 是否存在。文件缺失时，后续步骤都应该顺序跳过，而不是让 notebook 中断。


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import pandas as pd
from IPython.display import display

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

warnings.filterwarnings(
    "ignore",
    message="Pandas requires version '1.3.6' or newer of 'bottleneck'",
)

from credit_visable.config import load_settings
from credit_visable.data.load_data import load_application_train, summarize_table_availability
from credit_visable.features import (
    FEATURE_SET_NAMES as SUPPORTED_FEATURE_SET_NAMES,
    PreprocessingOptions,
    build_feature_set_manifest,
    prepare_preprocessing_artifacts,
    save_preprocessing_artifacts,
    select_feature_set_frame,
    split_feature_types,
)
from credit_visable.utils.paths import get_paths

settings = load_settings()
paths = get_paths()

RAW_DATA_DIR = paths.data_raw
PREPROCESSING_OUTPUT_ROOT = paths.data_processed / "preprocessing"
FEATURE_SET_NAMES = list(SUPPORTED_FEATURE_SET_NAMES)
APPLICATION_NROWS = None
MISSINGNESS_TOP_N = 12
FEATURE_NAME_PREVIEW = 12

table_summary = summarize_table_availability(data_dir=RAW_DATA_DIR, settings=settings)
available_table_names = set(table_summary.loc[table_summary["available"], "table_name"])
PHASE_2_READY = "application_train" in available_table_names


def summarize_missingness(frame: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "column": frame.columns,
            "missing_count": frame.isna().sum().values,
            "missing_rate": frame.isna().mean().values,
        }
    )
    return summary.sort_values(["missing_rate", "missing_count"], ascending=False).head(MISSINGNESS_TOP_N)


def build_matrix_summary(feature_set_name: str, matrix_name: str, matrix) -> dict[str, object]:
    total_cells = matrix.shape[0] * matrix.shape[1]
    density = float(matrix.nnz / total_cells) if total_cells else 0.0
    return {
        "feature_set": feature_set_name,
        "matrix": matrix_name,
        "rows": int(matrix.shape[0]),
        "columns": int(matrix.shape[1]),
        "nnz": int(matrix.nnz),
        "density": density,
    }


print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Preprocessing output root: {PREPROCESSING_OUTPUT_ROOT}")
print(f"Feature sets: {FEATURE_SET_NAMES}")
print(f"Phase 2 ready: {PHASE_2_READY}")
display(table_summary)


## 2. 主表加载与两套 feature set 清单

这里先用 `application_train` 构造两套固定数据口径，并把字段归类为 `traditional` / `proxy`。后续预处理、建模和对比都只能沿用这两套定义。


In [ ]:
application_train = None
feature_frames = {}
feature_set_manifests = {}
artifacts_by_set = {}
saved_artifacts_by_set = {}

if not PHASE_2_READY:
    print("Phase 2 已跳过，因为 application_train.csv 不存在。")
else:
    application_train = load_application_train(data_dir=RAW_DATA_DIR, nrows=APPLICATION_NROWS)
    overview_rows = []
    preview_rows = []

    for feature_set_name in FEATURE_SET_NAMES:
        manifest = build_feature_set_manifest(
            application_train.columns.tolist(),
            feature_set_name=feature_set_name,
            target_column=settings.target_column,
            id_column=settings.id_column,
        )
        feature_set_manifests[feature_set_name] = manifest
        feature_frames[feature_set_name] = select_feature_set_frame(
            application_train,
            feature_set_name=feature_set_name,
            target_column=settings.target_column,
            id_column=settings.id_column,
        )
        feature_groups = split_feature_types(
            feature_frames[feature_set_name],
            target_column=settings.target_column,
            id_column=settings.id_column,
        )

        overview_rows.append(
            {
                "feature_set": feature_set_name,
                "selected_feature_count": manifest["selected_feature_count"],
                "traditional_feature_count": manifest["traditional_feature_count"],
                "proxy_feature_count": manifest["proxy_feature_count"],
                "numeric_feature_count": len(feature_groups["numeric"]),
                "categorical_feature_count": len(feature_groups["categorical"]),
            }
        )
        preview_rows.append(
            {
                "feature_set": feature_set_name,
                "preview_type": "selected_features_preview",
                "preview": ", ".join(manifest["selected_feature_columns"][:FEATURE_NAME_PREVIEW]),
            }
        )
        preview_rows.append(
            {
                "feature_set": feature_set_name,
                "preview_type": "proxy_features_preview",
                "preview": ", ".join(manifest["proxy_feature_columns"][:FEATURE_NAME_PREVIEW]),
            }
        )

    print(f"application_train shape: {application_train.shape}")
    display(pd.DataFrame(overview_rows))
    display(pd.DataFrame(preview_rows))


## 3. 诊断摘要

这里保留 notebook 可视检查：每套 feature set 各自看缺失率摘要与字段族规模，确认 proxy 列只在 `traditional_plus_proxy` 里进入训练输入。


In [ ]:
if not PHASE_2_READY:
    print("诊断摘要已跳过，因为 application_train.csv 不存在。")
else:
    feature_family_rows = []
    missingness_frames = []

    for feature_set_name, feature_frame in feature_frames.items():
        feature_only_frame = feature_frame.drop(
            columns=[settings.target_column, settings.id_column],
            errors="ignore",
        )
        feature_groups = split_feature_types(
            feature_frame,
            target_column=settings.target_column,
            id_column=settings.id_column,
        )
        feature_family_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "feature_family": "numeric",
                    "column_count": len(feature_groups["numeric"]),
                },
                {
                    "feature_set": feature_set_name,
                    "feature_family": "categorical",
                    "column_count": len(feature_groups["categorical"]),
                },
            ]
        )
        missingness_summary = summarize_missingness(feature_only_frame)
        missingness_summary.insert(0, "feature_set", feature_set_name)
        missingness_frames.append(missingness_summary)

    display(pd.DataFrame(feature_family_rows))
    if missingness_frames:
        display(pd.concat(missingness_frames, ignore_index=True))


## 4. 拟合预处理并按 feature set 落盘

这一步默认一次性遍历两套 feature set。每套都会单独保存到 `data/processed/preprocessing/<feature_set>/`，并额外落一个 `feature_set_manifest.json` 说明字段归属。


In [ ]:
if not PHASE_2_READY:
    print("预处理拟合与落盘已跳过，因为 application_train.csv 不存在。")
else:
    options = PreprocessingOptions()
    split_rows = []
    matrix_rows = []
    artifact_rows = []

    for feature_set_name in FEATURE_SET_NAMES:
        feature_frame = feature_frames[feature_set_name]
        artifacts = prepare_preprocessing_artifacts(
            feature_frame,
            target_column=settings.target_column,
            id_column=settings.id_column,
            options=options,
        )
        artifacts_by_set[feature_set_name] = artifacts

        output_dir = PREPROCESSING_OUTPUT_ROOT / feature_set_name
        saved = save_preprocessing_artifacts(artifacts, output_dir=output_dir)
        feature_set_manifest_path = output_dir / "feature_set_manifest.json"
        feature_set_manifest_payload = {
            **feature_set_manifests[feature_set_name],
            "output_dir": str(output_dir.resolve()),
        }
        feature_set_manifest_path.write_text(
            json.dumps(feature_set_manifest_payload, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        saved["feature_set_manifest"] = feature_set_manifest_path
        saved_artifacts_by_set[feature_set_name] = saved

        split_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "split": "train",
                    "rows": len(artifacts.y_train),
                    "bad_rate": float(artifacts.y_train.mean()),
                    "feature_count": len(artifacts.feature_names),
                },
                {
                    "feature_set": feature_set_name,
                    "split": "valid",
                    "rows": len(artifacts.y_valid),
                    "bad_rate": float(artifacts.y_valid.mean()),
                    "feature_count": len(artifacts.feature_names),
                },
            ]
        )
        matrix_rows.extend(
            [
                build_matrix_summary(feature_set_name, "X_train", artifacts.X_train),
                build_matrix_summary(feature_set_name, "X_valid", artifacts.X_valid),
            ]
        )
        artifact_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "artifact": artifact_name,
                    "path": str(path),
                }
                for artifact_name, path in saved.items()
            ]
        )

    display(pd.DataFrame(split_rows))
    display(pd.DataFrame(matrix_rows))
    display(pd.DataFrame(artifact_rows))


## 5. Checkpoint

Phase 2 现在的成功标准不再是“一套矩阵可用”，而是两套 feature set 的标准产物都齐全，而且 train / valid 主键切分保持一致。


In [ ]:
if not PHASE_2_READY:
    print("Checkpoint: 当前仍停在 Phase 1 / Phase 2 交界，因为 application_train.csv 缺失。")
else:
    required_artifacts = [
        "X_train",
        "X_valid",
        "train_meta",
        "valid_meta",
        "feature_names",
        "manifest",
        "feature_set_manifest",
    ]
    validation_rows = []
    for feature_set_name in FEATURE_SET_NAMES:
        for artifact_name in required_artifacts:
            artifact_path = saved_artifacts_by_set[feature_set_name][artifact_name]
            validation_rows.append(
                {
                    "feature_set": feature_set_name,
                    "artifact": artifact_name,
                    "exists": artifact_path.exists(),
                    "path": str(artifact_path),
                }
            )

    split_alignment_ok = False
    if all(feature_set_name in artifacts_by_set for feature_set_name in FEATURE_SET_NAMES):
        base_feature_set = FEATURE_SET_NAMES[0]
        reference_train_ids = artifacts_by_set[base_feature_set].train_ids
        reference_valid_ids = artifacts_by_set[base_feature_set].valid_ids
        split_alignment_ok = True
        for feature_set_name in FEATURE_SET_NAMES[1:]:
            compared_artifacts = artifacts_by_set[feature_set_name]
            split_alignment_ok = split_alignment_ok and (
                reference_train_ids is not None
                and compared_artifacts.train_ids is not None
                and reference_valid_ids is not None
                and compared_artifacts.valid_ids is not None
                and reference_train_ids.tolist() == compared_artifacts.train_ids.tolist()
                and reference_valid_ids.tolist() == compared_artifacts.valid_ids.tolist()
            )

    phase_3_ready = all(row["exists"] for row in validation_rows) and split_alignment_ok
    display(pd.DataFrame(validation_rows))
    print(f"Phase 3 ready: {phase_3_ready}")
    print(f"Cross-feature-set split alignment ok: {split_alignment_ok}")
    if phase_3_ready:
        print("下一步可以在 Phase 3 同时训练 traditional_core 与 traditional_plus_proxy 的 logistic baseline。")
    else:
        print("当前还不能进入 Phase 3，请先检查上面的缺失产物或 split 对齐问题。")


In [ ]:
# Phase 2 report upgrade add-ons
from credit_visable.features import build_feature_catalog, build_preprocessing_decision_manifest
from credit_visable.utils import build_report_summary_fields, to_builtin

feature_catalog = pd.DataFrame()
preprocessing_decision_summary = pd.DataFrame()

if not PHASE_2_READY:
    print("Phase 2 report upgrade skipped because application_train.csv is unavailable.")
else:
    PREPROCESSING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    feature_catalog = build_feature_catalog(
        application_train,
        target_column=settings.target_column,
        id_column=settings.id_column,
    )
    feature_catalog_path = PREPROCESSING_OUTPUT_ROOT / "feature_catalog.csv"
    feature_catalog.to_csv(feature_catalog_path, index=False)

    decision_rows = []
    for feature_set_name in FEATURE_SET_NAMES:
        manifest = build_preprocessing_decision_manifest(
            application_train,
            feature_set_name=feature_set_name,
            target_column=settings.target_column,
            id_column=settings.id_column,
            feature_catalog=feature_catalog,
        )
        decision_rows.append(
            {
                "feature_set": feature_set_name,
                "selected_feature_count": manifest["selected_feature_count"],
                "numeric_feature_count": manifest["numeric_feature_count"],
                "categorical_feature_count": manifest["categorical_feature_count"],
                "proxy_feature_count": manifest["proxy_feature_count"],
            }
        )
        manifest_path = PREPROCESSING_OUTPUT_ROOT / feature_set_name / "preprocessing_decision_manifest.json"
        manifest_path.write_text(json.dumps(to_builtin(manifest), indent=2, ensure_ascii=False), encoding="utf-8")

    preprocessing_decision_summary = pd.DataFrame(decision_rows)
    preprocessing_decision_summary_path = PREPROCESSING_OUTPUT_ROOT / "preprocessing_decision_summary.csv"
    preprocessing_decision_summary.to_csv(preprocessing_decision_summary_path, index=False)

    summary_payload = {
        "phase": 2,
        "raw_data_dir": str(RAW_DATA_DIR),
        "feature_sets": FEATURE_SET_NAMES,
        "phase_3_ready": bool(phase_3_ready),
        "cross_feature_set_split_alignment_ok": bool(split_alignment_ok),
        "feature_catalog_path": str(feature_catalog_path),
        "preprocessing_decision_summary_path": str(preprocessing_decision_summary_path),
    }
    summary_payload.update(
        build_report_summary_fields(
            headline="Phase 2 documents feature inclusion, proxy status, and preprocessing decisions for both feature regimes.",
            key_findings=[
                "Feature catalog now links missingness, proxy status, IV, and feature-family membership.",
                "Each feature set now writes a preprocessing decision manifest alongside the existing matrix artifacts.",
                "The training matrices remain unchanged: this is a documentation and governance upgrade, not a modeling-regime change.",
            ],
            business_implications=[
                "Phase 3 and Phase 4 can now explain not only which features were used, but why they were allowed into each regime.",
                "Proxy counts and IV rankings provide an auditable bridge from EDA to model evaluation.",
                "The upgrade keeps the current sparse matrix pipeline intact, which avoids breaking downstream notebook assumptions.",
            ],
            figure_paths={},
        )
    )
    (PREPROCESSING_OUTPUT_ROOT / "processing_methods_summary.json").write_text(
        json.dumps(to_builtin(summary_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (PREPROCESSING_OUTPUT_ROOT / "processing_methods_summary_cn.md").write_text(
        "\n".join(
            [
                "# Phase 2 Processing Summary",
                "",
                f"- Feature sets: {', '.join(FEATURE_SET_NAMES)}",
                f"- Phase 3 ready: {phase_3_ready}",
                f"- Split alignment ok: {split_alignment_ok}",
                "- Added artifacts: feature_catalog.csv, preprocessing_decision_summary.csv, preprocessing_decision_manifest.json",
            ]
        ),
        encoding="utf-8",
    )
    display(feature_catalog.head(20))
    display(preprocessing_decision_summary)
